# Read a JSON file in Python with chDB (drop-in pandas)

Companion to [read-json-file-python](https://clickhouse.com/resources/engineering/read-json-file-python).

Run `./generate.sh` first to create `data/`. Requires `pip install chdb pandas`.

## 1. Read NDJSON into a DataFrame (types auto-inferred)

In [ ]:
import chdb.datastore as pd

df = pd.read_json("data/events.ndjson", lines=True)
df

In [ ]:
df.dtypes

## 2. Filter + aggregate the way you already do (pandas, not SQL)

In [ ]:
purchases = (df[df["event_type"] == "purchase"]
             .groupby("country")["revenue"].sum()
             .sort_values(ascending=False))
purchases

## 3. Nested objects: extract fields with .apply after .to_pandas()

In [ ]:
import pandas as real_pd

pdf = df.to_pandas()
pdf["user_id"] = pdf["user"].apply(lambda x: x[0])
pdf["user_tier"] = pdf["user"].apply(lambda x: x[1])
pdf[["event_id", "user_id", "user_tier", "revenue"]]

## 4. Nested arrays: flatten with .explode

In [ ]:
exploded = (pdf[["event_id", "items"]]
            .explode("items")
            .dropna(subset=["items"])
            .reset_index(drop=True))
exploded

## 5. A top-level JSON array reads without lines=True

In [ ]:
arr_df = pd.read_json("data/events_array.json")
len(arr_df)

## 6. Hand off to real pandas when you need it

In [ ]:
result = df[df["event_type"] == "purchase"].groupby("country")["revenue"].sum()
pdf_result = result.to_pandas()   # a real pandas.Series, in memory
type(pdf_result)

## 7. Performance: same code, one import swapped, on a 2M-row NDJSON

Apple M4 Pro (14 cores, 24 GB RAM, macOS), best-of-3 warm: ~12x faster than `pandas.read_json`.

In [ ]:
import time

def datastore_agg():
    d = pd.read_json("data/events_2m.ndjson", lines=True)
    r = (d[d["event_type"] == "purchase"]
         .groupby("country")["revenue"].sum()
         .sort_values(ascending=False))
    return r.to_pandas()

def pandas_agg():
    import pandas as real_pd2
    p = real_pd2.read_json("data/events_2m.ndjson", lines=True)
    return (p[p["event_type"] == "purchase"]
            .groupby("country")["revenue"].sum()
            .sort_values(ascending=False))

def best_of_3(fn):
    fn()
    best = float("inf")
    for _ in range(3):
        t0 = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t0)
    return best

pd_s = best_of_3(pandas_agg)
ds_s = best_of_3(datastore_agg)
print(f"import pandas as pd:            {pd_s:.3f}s")
print(f"import chdb.datastore as pd:    {ds_s:.3f}s")
print(f"speedup:                        {pd_s / ds_s:.1f}x")